# 08 多会话目录与旧会话恢复

**用途：** 验证会话目录和checkpoint分库、客户隔离、重命名、归档与重启续聊。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


In [2]:
from pathlib import Path
import agent_graph
from conversation_repository import ConversationRepository, ConversationAccessError
from handoff_repository import HandoffRepository
from memory_repository import MemoryRepository

temp_dir = tempfile.TemporaryDirectory()
root = Path(temp_dir.name)
catalog = ConversationRepository(root / "conversations.sqlite3")
saver = agent_graph.create_sqlite_checkpointer(root / "checkpoints.sqlite3")
graph = agent_graph.build_graph(
    saver,
    HandoffRepository(root / "handoff.sqlite3"),
    MemoryRepository(root / "memory.sqlite3"),
)
result_a = agent_graph.start_graph_agent(
    "小松PC200原厂液压泵要1件，有没有现货？",
    thread_id="thread-a",
    customer_id="customer-a",
    approval_mode="auto",
    graph=graph,
)
result_b = agent_graph.start_graph_agent(
    "卡特320D原厂液压泵要1件，有没有库存？",
    thread_id="thread-b",
    customer_id="customer-a",
    approval_mode="auto",
    graph=graph,
)
catalog.record_result(result_a, customer_id="customer-a")
catalog.record_result(result_b, customer_id="customer-a")
show_table(catalog.list_threads("customer-a"))
check_equal("同一客户有两个会话", len(catalog.list_threads("customer-a")), 2)

D:\new things\项目1\day1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,thread_id,customer_id,title,title_is_custom,status,channel,execution_mode,turn_count,last_message_preview,last_request_id,created_at,updated_at,archived_at,archived
0,thread-b,customer-a,卡特320D原厂液压泵要1件，有没有库存？,False,completed,web,LangGraph,1,卡特320D原厂液压泵要1件，有没有库存？,66f624088d71425ebdbdf35bc418c677,2026-07-28T06:34:28.182339+00:00,2026-07-28T06:34:28.182339+00:00,,False
1,thread-a,customer-a,小松PC200原厂液压泵要1件，有没有现货？,False,completed,web,LangGraph,1,小松PC200原厂液压泵要1件，有没有现货？,5d0e425768664ef2bf083664591c1d0f,2026-07-28T06:34:28.176340+00:00,2026-07-28T06:34:28.176340+00:00,,False


[PASS] 同一客户有两个会话 | actual=2, expected=2


{'检查项': '同一客户有两个会话', '状态': 'PASS', '说明': 'actual=2, expected=2'}

In [3]:
catalog.rename_thread("thread-a", customer_id="customer-a", title="PC200主泵")
catalog.archive_thread("thread-b", customer_id="customer-a")
active = catalog.list_threads("customer-a")
all_items = catalog.list_threads("customer-a", include_archived=True)
check_equal("归档后默认只显示一个", len(active), 1)
check_equal("显示归档时仍有两个", len(all_items), 2)

restarted_saver = agent_graph.create_sqlite_checkpointer(root / "checkpoints.sqlite3")
restarted_graph = agent_graph.build_graph(
    restarted_saver,
    HandoffRepository(root / "handoff.sqlite3"),
    MemoryRepository(root / "memory.sqlite3"),
)
loaded = agent_graph.load_graph_thread(
    "thread-a", customer_id="customer-a", graph=restarted_graph
)
check_equal("旧会话从checkpoint恢复", loaded["thread_id"], "thread-a")

blocked = False
try:
    catalog.get_thread("thread-a", customer_id="customer-b")
except ConversationAccessError:
    blocked = True
check("跨客户目录读取被拒绝", blocked)
saver.conn.close()
restarted_saver.conn.close()
temp_dir.cleanup()

[PASS] 归档后默认只显示一个 | actual=1, expected=1
[PASS] 显示归档时仍有两个 | actual=2, expected=2
[PASS] 旧会话从checkpoint恢复 | actual='thread-a', expected='thread-a'
[PASS] 跨客户目录读取被拒绝


In [4]:
session_tests = run_unittest(
    ["tests.test_conversation_sessions"],
    project2_root=PROJECT2_ROOT,
)
check("多会话6条通过", "Ran 6 tests" in session_tests.output and "OK" in session_tests.output)

$ D:\new things\项目1\day1\.venv\Scripts\python.exe -m unittest tests.test_conversation_sessions -v
test_checkpoint_load_rejects_missing_or_other_customer (tests.test_conversation_sessions.ConversationSessionTests.test_checkpoint_load_rejects_missing_or_other_customer) ... ok
test_customer_access_is_enforced_for_all_mutations (tests.test_conversation_sessions.ConversationSessionTests.test_customer_access_is_enforced_for_all_mutations) ... ok
test_old_checkpoint_loads_and_continues_after_graph_restart (tests.test_conversation_sessions.ConversationSessionTests.test_old_checkpoint_loads_and_continues_after_graph_restart) ... ok
test_record_result_creates_auto_title_and_preview (tests.test_conversation_sessions.ConversationSessionTests.test_record_result_creates_auto_title_and_preview) ... ok
test_rename_archive_and_restore (tests.test_conversation_sessions.ConversationSessionTests.test_rename_archive_and_restore) ... ok
test_threads_are_recent_first_and_customer_scoped (tests.test_conversat

{'检查项': '多会话6条通过', '状态': 'PASS', '说明': ''}

## 两个SQLite的职责

- `conversation_threads.sqlite3`：标题、列表、状态、预览、归档。
- `langgraph_checkpoints.sqlite3`：State、消息、摘要、槽位、interrupt和恢复。

当前能按客户列出和恢复，不支持对话全文搜索。Streamlit Cloud没有外部持久数据库时，不能承诺重新部署后本地SQLite永久保留。

### 面试官会问

1. checkpoint能“查找历史”到什么程度？
2. 为什么不直接读取LangGraph内部SQLite表？
3. 旧会话如何防止跨客户串线？
4. 最多能记忆几轮？目录记录和模型上下文有什么区别？
5. 生产环境如何迁移到Postgres和租户鉴权？

### 参考答案

1. **checkpoint能查历史到什么程度？** 已知`thread_id`时可以读取最新State和逐步StateSnapshot历史，也能恢复等待中的interrupt；它不是面向客户的搜索引擎，当前不支持按对话全文或关键词全局搜索。
2. **为什么不直接读LangGraph内部表？** 内部表结构属于框架实现，版本升级可能变化，直接SQL也容易绕过反序列化和权限检查。项目通过`get_state/get_state_history`和封装的`load_graph_thread`使用公开接口。
3. **如何防跨客户串线？** 会话目录的查询、重命名、归档和恢复都必须携带`customer_id`；加载checkpoint后再次比较State里的客户归属。目录层和State层任一不一致都会拒绝。
4. **最多记忆几轮？** 默认模型上下文保留8条近期消息，约4轮完整问答，更早内容进入滚动摘要，因此会话轮数没有固定硬上限，但不会永久逐字送入模型。会话目录只保存标题、预览、状态等产品元数据，不等于模型记忆。
5. **生产环境怎样迁移？** 把checkpointer、会话目录、长期记忆和服务单迁到Postgres，所有表带`tenant_id/customer_id`并建立索引；API层从登录态推导租户，不能相信前端传入值，同时加入行级权限、加密、保留/删除策略、备份和审计。

**代码落点：** `conversation_repository.py`、`agent_graph.py::load_graph_thread`、`tests/test_conversation_sessions.py`。